Clase 010 · `010_importar_formatos` · Último archivo del recorrido:
`v6_sucio_tipos.csv` junta varios problemas a la vez. Los nombres de columna
traen espacios y mayúsculas, la fecha viene en formato de 12 horas con AM/PM,
los faltantes están marcados con palabras y hay columnas numéricas que llegan
como texto. Todo se resuelve en una sola celda, paso por paso.

::: {.callout-important title="Datos"}
El archivo `v6_sucio_tipos.csv` no viaja en el repositorio; colócalo en
`clases/010_importar_formatos/data/001_raw/` (se comparte en clase).
:::

## Librerías

In [1]:
import pandas as pd

## Leer, renombrar, fechar y convertir tipos

La celda hace cinco cosas en orden:

1. **Diccionario de nombres.** `nombres` traduce los encabezados sucios a nombres limpios. Fíjate en
   las llaves: `" dhi"` con espacio al inicio y `'GHI '` con espacio al final. Así vienen en el
   archivo, y si la llave no coincide carácter por carácter, `rename` no hace nada con esa columna.
2. **Leer con centinelas de texto.** Igual que en `v4_centinelas.csv`, `na_values` convierte en `NaN`
   las marcas `"error"`, `" "` y `"sensor off"`.
3. **Renombrar.** `rename(columns=nombres, inplace=True)` cambia solo las cuatro columnas del
   diccionario; las demás conservan su nombre original.
4. **Construir el índice de tiempo.** La columna `tiempo` trae horas como `01:05:00 PM`. En vez de
   dejar que `pandas` adivine, `format="%Y-%m-%d %I:%M:%S %p"` describe exactamente el patrón: `%I`
   es la hora en reloj de 12 y `%p` el AM/PM. Luego `set_index` la vuelve índice.
5. **Convertir tipos.** `ghi` llegó como texto porque usa coma decimal: primero `str.replace(",",".")`
   y después `astype(float)`. `p` solo necesita `astype(float)`, porque lo que la volvía texto eran
   los centinelas que ya quitamos en el paso 2.

`info()` muestra hasta dónde llegamos: índice `DatetimeIndex` completo, `dhi`, `ghi` y `p` ya en
`float64`, y la presión con los faltantes que esperábamos. Pero quedan tres columnas como texto
(`HUMEDAD  REL`, ` Temp Bulbo Seco` y `vel.viento`) y seis nombres sin limpiar. Ahí se quedó la
clase: el mismo tratamiento de los pasos 1 y 5, aplicado a esas columnas, termina la limpieza.

In [2]:
f = "../data/001_raw/v6_sucio_tipos.csv"
nombres = {
    "Fecha Hora":"tiempo",
    " dhi":"dhi",
    'GHI ':"ghi",
    'Presion Atm':"p"
}
esolmet = pd.read_csv(f,na_values=["error"," ","sensor off"])
esolmet.rename(columns=nombres,inplace=True)
esolmet.tiempo = pd.to_datetime(esolmet.tiempo,format="%Y-%m-%d %I:%M:%S %p")
esolmet.set_index("tiempo",inplace=True)
esolmet.ghi = esolmet.ghi.str.replace(",",".").astype(float)
esolmet.p = esolmet.p.astype(float)

esolmet.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 172433 entries, 2026-04-08 00:00:00 to 2026-08-05 18:04:00
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   dhi                172433 non-null  float64
 1   DNI                172433 non-null  float64
 2   ghi                172431 non-null  float64
 3   p                  142961 non-null  float64
 4   Lluvia acum        172433 non-null  float64
 5   HUMEDAD  REL       171894 non-null  str    
 6    Temp Bulbo Seco   172433 non-null  str    
 7   UV                 172433 non-null  float64
 8   Dir Viento         172433 non-null  float64
 9   vel.viento         172433 non-null  str    
dtypes: float64(7), str(3)
memory usage: 18.3 MB
